In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

c:\Users\sabri\miniconda3\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [14]:

import os

# Create folder for product images
os.makedirs("product_images", exist_ok=True)

# Create the name column
products_df["name"] = products_df["productDisplayName"]

# Assign placeholder prices
price_map = {
    "Apparel": 29.99,
    "Accessories": 24.99,
    "Footwear": 59.99,
    "Personal Care": 14.99
}

products_df["price"] = (
    products_df["masterCategory"]
    .map(price_map)
    .fillna(19.99)
)

# Save images and create image paths
image_paths = []

for _, product in products_df.iterrows():
    image_path = f"product_images/{product['id']}.jpg"

    product["image"].convert("RGB").save(image_path)

    image_paths.append(image_path)

products_df["image_path"] = image_paths

# Verify
print(products_df[
    ["name", "price", "masterCategory", "image_path"]
].head())


                                            name  price masterCategory  \
0               Turtle Check Men Navy Blue Shirt  29.99        Apparel   
1             Peter England Men Party Blue Jeans  29.99        Apparel   
2                       Titan Women Silver Watch  24.99    Accessories   
3  Manchester United Men Solid Black Track Pants  29.99        Apparel   
4                          Puma Men Grey T-shirt  29.99        Apparel   

                 image_path  
0  product_images/15970.jpg  
1  product_images/39386.jpg  
2  product_images/59263.jpg  
3  product_images/21379.jpg  
4  product_images/53759.jpg  


**Step 3: Encoding Images for API**
<br>Objective: Convert product images to base64 format for API transmission.

In [15]:
import base64
from io import BytesIO
 
def encode_image_to_base64(image):
    """Encode a PIL image or image file path to a base64 string."""
    if isinstance(image, (str, Path)):
        with open(image, "rb") as img_file:
            image_bytes = img_file.read()
    else:
        image_buffer = BytesIO()
        image.save(image_buffer, format=image.format or "JPEG")
        image_bytes = image_buffer.getvalue()
 
    return base64.b64encode(image_bytes).decode("utf-8")

 

In [16]:
# Example usage
image_column = "image" if "image" in products_df.columns else "image_path"
sample_image = products_df.iloc[0][image_column]
encoded_image = encode_image_to_base64(sample_image)
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


**Step 4: Creating the Product Listing Prompt**
<br>Objective: Design an effective prompt for generating product listings.
<br>What to do:
<br>Create a prompt template
<br>Include product metadata
<br>Specify the desired output format

In [17]:

def create_product_prompt(name, price, category, additional_info=""):
    prompt = f"""
You are an expert e-commerce copywriter.

Create an accurate, engaging, SEO-friendly product listing
based on the product information and attached image.

PRODUCT INFORMATION:
- Name: {name}
- Price: ${price:.2f}
- Category: {category}
- Additional information: {additional_info}

INSTRUCTIONS:
1. Create a clear, attractive product title.
2. Write a compelling product description.
3. List 5-7 key product features.
4. Generate 10-15 relevant SEO keywords.
5. Use the product image to identify visible details.
6. Do not invent materials, specifications, or features
   that cannot be verified.

Return ONLY valid JSON using this structure:

{{
    "title": "Product title",
    "description": "Product description",
    "features": [
        "Feature 1",
        "Feature 2"
    ],
    "keywords": [
        "keyword 1",
        "keyword 2"
    ]
}}

Do not include explanations or markdown formatting.
"""
    return prompt


In [18]:

test_prompt = create_product_prompt(
    name="Men's Navy Blue T-Shirt",
    price=29.99,
    category="Apparel",
    additional_info="Casual wear, summer collection"
)

print(test_prompt)



You are an expert e-commerce copywriter.

Create an accurate, engaging, SEO-friendly product listing
based on the product information and attached image.

PRODUCT INFORMATION:
- Name: Men's Navy Blue T-Shirt
- Price: $29.99
- Category: Apparel
- Additional information: Casual wear, summer collection

INSTRUCTIONS:
1. Create a clear, attractive product title.
2. Write a compelling product description.
3. List 5-7 key product features.
4. Generate 10-15 relevant SEO keywords.
5. Use the product image to identify visible details.
6. Do not invent materials, specifications, or features
   that cannot be verified.

Return ONLY valid JSON using this structure:

{
    "title": "Product title",
    "description": "Product description",
    "features": [
        "Feature 1",
        "Feature 2"
    ],
    "keywords": [
        "keyword 1",
        "keyword 2"
    ]
}

Do not include explanations or markdown formatting.



In [25]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Client initialized successfully")

Client initialized successfully


### Step 5: Call ChatGPT with the image
Send **one message with two parts**:

In [26]:

import json

def generate_product_listing(name, price, category, image_path, additional_info=""):

    # Step 1: Build the prompt
    prompt = create_product_prompt(
        name, price, category, additional_info
    )

    # Step 2: Encode the image
    base64_image = encode_image_to_base64(image_path)

    # Step 3: Call the OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ]
    )

    # Step 4: Extract the generated text
    result = response.choices[0].message.content

    # Step 5: Remove markdown formatting if present
    result = result.strip()

    if result.startswith("```"):
        result = result.removeprefix("```json")
        result = result.removeprefix("```")
        result = result.removesuffix("```").strip()

    # Step 6: Convert JSON text into a Python dictionary
    return json.loads(result)


In [27]:
print(products_df.columns.tolist())

['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image', 'name', 'price', 'image_path']


In [28]:


product = products_df.iloc[0]

listing = generate_product_listing(
    name=product["name"],
    price=product["price"],
    category=product["masterCategory"],
    image_path=product["image_path"]
)

print(json.dumps(listing, indent=2))



{
  "title": "Turtle Check Men Navy Blue Shirt",
  "description": "Enhance your wardrobe with the Turtle Check Men Navy Blue Shirt, designed for style and comfort. This versatile shirt features a contemporary check pattern that effortlessly blends casual and sophisticated looks. Perfect for any occasion, whether it\u2019s a day at the office or a night out with friends, this shirt will keep you looking sharp and feeling great.",
  "features": [
    "Stylish navy blue check pattern",
    "Short sleeves for breathability",
    "Comfortable fit for all-day wear",
    "Easy to pair with jeans or chinos",
    "Lightweight and durable fabric",
    "Great for both casual and semi-formal events",
    "Available in multiple sizes"
  ],
  "keywords": [
    "men's shirt",
    "navy blue shirt",
    "checked shirt",
    "casual shirt",
    "fashion apparel",
    "stylish men's clothing",
    "short sleeve shirt",
    "comfortable fit shirt",
    "men's fashion",
    "versatile men's apparel",
    

### Step 6: Process many products
- Loop over 5 products (start small to save credits)
- Wrap each call in `try / except`, so **one failure doesn't stop the loop**
- Wait 1 second between calls with `time.sleep(1)` to avoid rate-limit errors
- Save everything to `generated_listings.json`

In [29]:

import json
import time

generated_listings = []

# Process the first 5 products
for index, product in products_df.head(5).iterrows():

    try:
        print(f"Processing product {index + 1}: {product['name']}")

        # Generate the listing using Step 5
        listing = generate_product_listing(
            name=product["name"],
            price=product["price"],
            category=product["masterCategory"],
            image_path=product["image_path"]
        )

        # Add product ID for traceability
        listing["product_id"] = int(product["id"])

        # Save successful result
        generated_listings.append(listing)

        print("Success!")

    except Exception as e:
        print(f"Error processing product {index + 1}: {e}")

    # Wait between API calls
    time.sleep(1)

# Save all successful listings to JSON
with open("generated_listings.json", "w", encoding="utf-8") as f:
    json.dump(generated_listings, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(generated_listings)} listings.")


Processing product 1: Turtle Check Men Navy Blue Shirt
Success!
Processing product 2: Peter England Men Party Blue Jeans
Success!
Processing product 3: Titan Women Silver Watch
Success!
Processing product 4: Manchester United Men Solid Black Track Pants
Success!
Processing product 5: Puma Men Grey T-shirt
Success!

Saved 5 listings.


In [30]:
with open("generated_listings.json", "r", encoding="utf-8") as f:
    saved_listings = json.load(f)

print("Total listings:", len(saved_listings))

for listing in saved_listings:
    print(listing["product_id"], "-", listing["title"])

Total listings: 5
15970 - Turtle Check Men Navy Blue Shirt - Stylish and Comfortable
39386 - Peter England Men Party Blue Jeans - Stylish & Comfortable
59263 - Titan Women Silver Watch - Elegant & Stylish Timepiece
21379 - Manchester United Solid Black Track Pants for Men
53759 - Puma Men's Grey T-Shirt - Stylish and Comfortable


### Step 7: Show error handling
Run the three small examples in the notebook and take screenshots:

| Example | What you'll see |
|---|---|
| Image file doesn't exist | `FileNotFoundError` is caught with a clear message |
| Fake API key | `AuthenticationError` is caught |
| JSON wrapped in ```` ```json ```` | Fails without cleaning, works after cleaning |


In [31]:
#Test 1 — Missing image file

try:
    encode_image_to_base64("product_images/nonexistent.jpg")

except FileNotFoundError as e:
    print("Missing image handled correctly.")
    print("Error:", e)

Missing image handled correctly.
Error: [Errno 2] No such file or directory: 'product_images/nonexistent.jpg'


In [32]:
#Test 2 — Invalid API key

from openai import OpenAI, AuthenticationError

fake_client = OpenAI(api_key="sk-invalid-test-key")

try:
    fake_client.models.list()

except AuthenticationError:
    print("Authentication error handled correctly.")

except Exception as e:
    print("Other API error:", type(e).__name__, str(e))

Authentication error handled correctly.


In [33]:
#Test 3 — JSON wrapped in Markdown

import json

test_response = '''```json
{"title": "Navy Blue Shirt"}
```'''

cleaned = test_response.strip()

if cleaned.startswith("```"):
    cleaned = cleaned.removeprefix("```json")
    cleaned = cleaned.removeprefix("```")
    cleaned = cleaned.removesuffix("```").strip()

parsed = json.loads(cleaned)

print(parsed)

{'title': 'Navy Blue Shirt'}
